# Uber Fare Prediction using Machine Learning

**Code by Dr. Rahul M. Dhokane, PRAVARA SVIT, NASHIK**

This notebook predicts Uber fares using **Linear Regression** and **Random Forest Regression**.

### Dataset
Place `uber.csv` in the same folder as this notebook, or update the dataset path in the loading cell.


## 1. Import required libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Display settings
pd.set_option("display.max_columns", None)


## 2. Define the Haversine distance function

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate spherical distance in kilometers between two GPS points."""
    earth_radius_km = 6371.0

    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))
    return earth_radius_km * c


## 3. Load the dataset

In [ ]:
# Google Colab path:
# df = pd.read_csv('/content/sample_data/uber.csv')

# Local Jupyter / VS Code path (uber.csv in the notebook folder):
df = pd.read_csv('uber.csv')

print("Dataset loaded successfully!")
print("Initial records:", len(df))
display(df.head())


## 4. Clean missing values and extract features

In [ ]:
required_columns = [
    'fare_amount',
    'pickup_datetime',
    'pickup_longitude',
    'pickup_latitude',
    'dropoff_longitude',
    'dropoff_latitude',
    'passenger_count'
]

missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

df = df.dropna(subset=required_columns).copy()

df['pickup_datetime'] = pd.to_datetime(
    df['pickup_datetime'], errors='coerce'
)
df = df.dropna(subset=['pickup_datetime']).copy()

df['hour'] = df['pickup_datetime'].dt.hour
df['day'] = df['pickup_datetime'].dt.day
df['month'] = df['pickup_datetime'].dt.month
df['year'] = df['pickup_datetime'].dt.year
df['dayofweek'] = df['pickup_datetime'].dt.dayofweek

df['distance_km'] = haversine_distance(
    df['pickup_latitude'],
    df['pickup_longitude'],
    df['dropoff_latitude'],
    df['dropoff_longitude']
)

print("Feature extraction completed.")
display(df.head())


## 5. Remove invalid records and outliers

In [ ]:
initial_count = len(df)

# Keep plausible fare amounts
df = df[df['fare_amount'].between(2.5, 500)]

# Keep coordinates within the approximate New York City region
df = df[
    df['pickup_latitude'].between(40.0, 42.0)
    & df['pickup_longitude'].between(-75.0, -72.0)
    & df['dropoff_latitude'].between(40.0, 42.0)
    & df['dropoff_longitude'].between(-75.0, -72.0)
]

# Keep plausible passenger counts and trip distances
df = df[df['passenger_count'].between(1, 6)]
df = df[df['distance_km'].between(0.1, 100)]

print(f"Cleaned {initial_count - len(df)} rows.")
print("Remaining records:", len(df))

if df.empty:
    raise ValueError("No records remain after cleaning. Check the dataset and filtering rules.")


## 6. Correlation analysis

In [ ]:
features = [
    'passenger_count',
    'hour',
    'day',
    'month',
    'year',
    'dayofweek',
    'distance_km'
]

corr = df[['fare_amount'] + features].corr(numeric_only=True)

print("Feature correlation with fare amount:")
display(corr['fare_amount'].sort_values(ascending=False).to_frame())

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()


## 7. Split data into training and testing sets

In [ ]:
X = df[features]
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


## 8. Train Linear Regression model

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

print("Linear Regression training completed.")


## 9. Train Random Forest Regression model

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest training completed.")


## 10. Evaluate model performance

In [ ]:
def get_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2

mae_lr, mse_lr, rmse_lr, r2_lr = get_metrics(y_test, y_pred_lr)
mae_rf, mse_rf, rmse_rf, r2_rf = get_metrics(y_test, y_pred_rf)

results = pd.DataFrame({
    'Metric': ['MAE ($)', 'MSE ($²)', 'RMSE ($)', 'R² Score'],
    'Linear Regression': [mae_lr, mse_lr, rmse_lr, r2_lr],
    'Random Forest': [mae_rf, mse_rf, rmse_rf, r2_rf]
})

print("MODEL EVALUATION")
display(results.round(6))


## 11. Compare model metrics

In [ ]:
x = np.arange(len(results['Metric']))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar(
    x - width / 2,
    results['Linear Regression'],
    width,
    label='Linear Regression'
)
plt.bar(
    x + width / 2,
    results['Random Forest'],
    width,
    label='Random Forest'
)

plt.xlabel('Evaluation Metrics')
plt.ylabel('Metric Values')
plt.title('Linear Regression vs Random Forest')
plt.xticks(x, results['Metric'], rotation=15)
plt.legend()
plt.tight_layout()
plt.show()


## 12. Actual vs predicted fare (Random Forest)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_rf, alpha=0.5)

min_value = min(y_test.min(), y_pred_rf.min())
max_value = max(y_test.max(), y_pred_rf.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    'r--',
    label='Ideal prediction'
)

plt.xlabel('Actual Fare ($)')
plt.ylabel('Predicted Fare ($)')
plt.title('Actual vs Predicted Uber Fare - Random Forest')
plt.legend()
plt.tight_layout()
plt.show()


## 13. Final result

In [ ]:
if r2_rf > r2_lr:
    best_model = "Random Forest Regression"
    best_r2 = r2_rf
else:
    best_model = "Linear Regression"
    best_r2 = r2_lr

print("FINAL RESULT")
print("Best Model:", best_model)
print(f"R² Score: {best_r2:.4f}")
print("Uber Fare Prediction Completed Successfully!")


## Conclusion

In this experiment, Uber fare prediction was performed using Linear Regression and Random Forest Regression. The dataset was cleaned, useful date/time and trip-distance features were extracted, and both models were trained and evaluated using MAE, MSE, RMSE, and R² score. The model with the higher R² score performed better on the test set.

**Note:** Results depend on the exact dataset and preprocessing. The geographic filtering rules are approximate New York City bounds and may need adjustment for a different dataset.
